In [ ]:
from pathlib import Path
import pandas as pd

data_folder = Path(r"C:\Users\20241553\Documents\crime_project\stop and search data")

csv_files = list(data_folder.rglob("*.csv"))

print("CSV files found:", len(csv_files))
for f in csv_files[:10]:
    print(f)

CSV files found: 36
C:\Users\20241553\Documents\crime_project\stop and search data\2023-04\2023-04-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-05\2023-05-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-06\2023-06-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-07\2023-07-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-08\2023-08-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-09\2023-09-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-10\2023-10-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-11\2023-11-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-

In [ ]:
stop_search_files = [
    f for f in csv_files 
    if "stop-and-search" in f.name.lower()
]

print("Stop/search files found:", len(stop_search_files))
for f in stop_search_files[:10]:
    print(f)

Stop/search files found: 36
C:\Users\20241553\Documents\crime_project\stop and search data\2023-04\2023-04-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-05\2023-05-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-06\2023-06-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-07\2023-07-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-08\2023-08-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-09\2023-09-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-10\2023-10-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search data\2023-11\2023-11-city-of-london-stop-and-search.csv
C:\Users\20241553\Documents\crime_project\stop and search da

In [ ]:
frames = []

for file in stop_search_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    temp["source_month_folder"] = file.parent.name
    frames.append(temp)

ss = pd.concat(frames, ignore_index=True)

print(ss.shape)
ss.head()

(6423, 17)


,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing,source_file,source_month_folder
0,Person search,2023-04-01T13:07:55+00:00,NaN,NaN,51.519045,-0.097078,Male,over 34,White - Any other White background,White,Police and Criminal Evidence Act 1984 (section 1),Offensive weapons,A no further action disposal,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04
1,Person and Vehicle search,2023-04-01T19:50:23+00:00,NaN,NaN,51.516360,-0.082993,Male,25-34,White - Any other White background,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04
2,Person search,2023-04-01T21:38:22+00:00,NaN,NaN,51.516869,-0.081113,Male,18-24,White - English/Welsh/Scottish/Northern Irish/...,White,Police and Criminal Evidence Act 1984 (section 1),Articles for use in criminal damage,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04
3,Person search,2023-04-02T14:14:59+00:00,NaN,NaN,51.516869,-0.081113,Male,over 34,Other ethnic group - Not stated,Black,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,False,False,2023-04-city-of-london-stop-and-search.csv,2023-04
4,Person search,2023-04-02T18:46:52+00:00,NaN,NaN,NaN,NaN,Male,18-24,Mixed/Multiple ethnic groups - Any other Mixed...,Black,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,False,False,2023-04-city-of-london-stop-and-search.csv,2023-04


In [ ]:
# Keep only rows with usable coordinates
ss = ss.dropna(subset=["Longitude", "Latitude"]).copy()

# Convert Longitude/Latitude to numeric, just in case
ss["Longitude"] = pd.to_numeric(ss["Longitude"], errors="coerce")
ss["Latitude"] = pd.to_numeric(ss["Latitude"], errors="coerce")

ss = ss.dropna(subset=["Longitude", "Latitude"]).copy()

# Create point geometry
ss_gdf = gpd.GeoDataFrame(
    ss,
    geometry=gpd.points_from_xy(ss["Longitude"], ss["Latitude"]),
    crs="EPSG:4326"
)

print(ss.shape)
ss_gdf.head()


(5063, 17)


,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing,source_file,source_month_folder,geometry
0,Person search,2023-04-01T13:07:55+00:00,NaN,NaN,51.519045,-0.097078,Male,over 34,White - Any other White background,White,Police and Criminal Evidence Act 1984 (section 1),Offensive weapons,A no further action disposal,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (-0.09708 51.51904)
1,Person and Vehicle search,2023-04-01T19:50:23+00:00,NaN,NaN,51.516360,-0.082993,Male,25-34,White - Any other White background,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (-0.08299 51.51636)
2,Person search,2023-04-01T21:38:22+00:00,NaN,NaN,51.516869,-0.081113,Male,18-24,White - English/Welsh/Scottish/Northern Irish/...,White,Police and Criminal Evidence Act 1984 (section 1),Articles for use in criminal damage,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (-0.08111 51.51687)
3,Person search,2023-04-02T14:14:59+00:00,NaN,NaN,51.516869,-0.081113,Male,over 34,Other ethnic group - Not stated,Black,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,False,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (-0.08111 51.51687)
7,Person search,2023-04-03T11:07:19+00:00,NaN,NaN,51.512105,-0.085089,Male,over 34,Other ethnic group - Not stated,White,Police and Criminal Evidence Act 1984 (section 1),Stolen goods,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (-0.08509 51.5121)


In [ ]:
lsoa = gpd.read_file("Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_-6970154227154374572.gpkg")

print(lsoa.shape)
lsoa.head()

(35672, 9)


,LSOA21CD,LSOA21NM,LSOA21NMW,BNG_E,BNG_N,LAT,LONG,GlobalID,geometry
0,E01000001,City of London 001A,,532123,181632,51.518169,-0.097150,{86214465-5CF4-4E8F-9492-3667471C42D6},"MULTIPOLYGON (((532105.312 182010.574, 532104...."
1,E01000002,City of London 001B,,532480,181715,51.518829,-0.091970,{CD40C491-6567-405F-8C18-426E17B356CE},"MULTIPOLYGON (((532634.497 181926.016, 532572...."
2,E01000003,City of London 001C,,532239,182033,51.521740,-0.095330,{7FD27AAF-D858-4E46-9099-92B43F66B948},"MULTIPOLYGON (((532135.138 182198.131, 532071...."
3,E01000005,City of London 001E,,533581,181283,51.514690,-0.076280,{7E76A16A-028F-4F49-84B5-6E5A67322F3C},"MULTIPOLYGON (((533808.018 180767.774, 533842...."
4,E01000006,Barking and Dagenham 016A,,544994,184274,51.538750,0.089317,{25AB047E-6FCF-4F76-9176-E92E44C0E097},"MULTIPOLYGON (((545122.049 184314.931, 545118...."


In [ ]:
print("Stop/search CRS:", ss_gdf.crs)
print("LSOA CRS:", lsoa.crs)

ss_gdf = ss_gdf.to_crs(lsoa.crs)

Stop/search CRS: EPSG:4326
LSOA CRS: EPSG:27700


In [17]:
ss_lsoa = gpd.sjoin(
    ss_gdf,
    lsoa[["LSOA21CD", "LSOA21NM", "geometry"]],
    how="left",
    predicate="within"
)

print(ss_lsoa.shape)
ss_lsoa.head()

(5063, 21)


,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,...,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing,source_file,source_month_folder,geometry,index_right,LSOA21CD,LSOA21NM
0,Person search,2023-04-01T13:07:55+00:00,NaN,NaN,51.519045,-0.097078,Male,over 34,White - Any other White background,White,...,Offensive weapons,A no further action disposal,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (532125.244 181729.801),0.0,E01000001,City of London 001A
1,Person and Vehicle search,2023-04-01T19:50:23+00:00,NaN,NaN,51.516360,-0.082993,Male,25-34,White - Any other White background,White,...,Controlled drugs,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (533110.229 181456.743),31058.0,E01032739,City of London 001F
2,Person search,2023-04-01T21:38:22+00:00,NaN,NaN,51.516869,-0.081113,Male,18-24,White - English/Welsh/Scottish/Northern Irish/...,White,...,Articles for use in criminal damage,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (533239.178 181516.767),31058.0,E01032739,City of London 001F
3,Person search,2023-04-02T14:14:59+00:00,NaN,NaN,51.516869,-0.081113,Male,over 34,Other ethnic group - Not stated,Black,...,Controlled drugs,A no further action disposal,False,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (533239.178 181516.767),31058.0,E01032739,City of London 001F
7,Person search,2023-04-03T11:07:19+00:00,NaN,NaN,51.512105,-0.085089,Male,over 34,Other ethnic group - Not stated,White,...,Stolen goods,Arrest,True,False,2023-04-city-of-london-stop-and-search.csv,2023-04,POINT (532977.196 180979.771),31058.0,E01032739,City of London 001F
